In [20]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import matplotlib.dates as mdates

files = [
    "/Users/macuser/Downloads/Weeks3&4Exercise/2022.01_WAVES-ACCESS-RECORDS.csv",
    "/Users/macuser/Downloads/Weeks3&4Exercise/2022.02_WAVES-ACCESS-RECORDS.csv",
    "/Users/macuser/Downloads/Weeks3&4Exercise/2022.03_WAVES-ACCESS-RECORDS-.csv",
    "/Users/macuser/Downloads/Weeks3&4Exercise/2022.04_WAVES-ACCESS-RECORDS.csv",
    "/Users/macuser/Downloads/Weeks3&4Exercise/2022.05-WAVES-ACCESS-RECORDS.csv",

]

dfs = []
for f in files:
    df = pd.read_csv(f, dtype=str, low_memory=False)
    df["source_file"] = Path(f).name
    dfs.append(df)

visits = pd.concat(dfs, ignore_index=True)
visits.head

<bound method NDFrame.head of           NAMELAST    NAMEFIRST NAMEMID     UIN  BDGNBR ACCESS_TYPE  \
0           MALLEY       ROBERT       N  U38641  183020          VA   
1           FARLEY  CHRISTOPHER       B  U38643     NaN          VA   
2         KALAMBUR        GUHAN       R  U38636  176380          VA   
3          KUKLISH      MATILDA       N  U38642  178294          VA   
4           SIERRA       RONNEY       F  U38598     NaN          VA   
...            ...          ...     ...     ...     ...         ...   
43029     ZURBRUGG      LINDSEY       A  U48414     NaN          VA   
43030  ZURBURGKING      HEATHER       N  U48460     NaN          VA   
43031    ZVIRBLYTE     AURELIJA       N  U51079     NaN          VA   
43032         ZYCH         KYLE       W  U48414     NaN          VA   
43033         TRUE        PETER       W  U48378     NaN          VA   

                       TOA  POA  TOD  POD  ... VISITEE_NAMEFIRST MEETING_LOC  \
0           1/1/2022 12:22  NaN  NaN 

In [22]:
# Numeric
if "TOTAL_PEOPLE" in visits.columns:
    visits["TOTAL_PEOPLE"] = pd.to_numeric(visits["TOTAL_PEOPLE"], errors="coerce")

# Parse core dates
def to_dt(s):
    return pd.to_datetime(s, errors="coerce")

for c in ["APPT_MADE_DATE", "APPT_START_DATE", "APPT_END_DATE", "APPT_CANCEL_DATE", "RELEASEDATE", "TOA", "TOD"]:
    if c in visits.columns:
        visits[c] = to_dt(visits[c])

# Use arrival time when available (TOA). Fallback to appointment start date (date-only) at noon.
visits["visit_start"] = visits["TOA"].copy()
fallback = visits["APPT_START_DATE"]
visits.loc[visits["visit_start"].isna(), "visit_start"] = fallback.loc[visits["visit_start"].isna()] + pd.Timedelta(hours=12)

visits["visit_date"] = visits["visit_start"].dt.date
visits["visit_dow"] = visits["visit_start"].dt.day_name()
visits["visit_hour"] = visits["visit_start"].dt.hour

# Canceled?
visits["canceled"] = visits["APPT_CANCEL_DATE"].notna() if "APPT_CANCEL_DATE" in visits.columns else False

# Lead time
visits["lead_days"] = (visits["APPT_START_DATE"] - visits["APPT_MADE_DATE"]).dt.total_seconds() / (24*3600)

# Time blocks based on visit_hour (TOA when present; otherwise noon fallback)
def time_block(h):
    if pd.isna(h):
        return np.nan
    h = int(h)
    if h < 10:
        return "Early (before 10am)"
    elif h < 12:
        return "Late morning (10–11:59)"
    elif h < 14:
        return "Midday (12–1:59)"
    elif h < 16:
        return "Afternoon (2–3:59)"
    else:
        return "Late day (4pm+)"
visits["time_block"] = visits["visit_hour"].map(time_block)

# School-friendly scope
wk = ["Monday","Tuesday","Wednesday","Thursday","Friday"]
visits_school = visits[visits["visit_dow"].isin(wk)].copy()

# Save cleaned
clean_path = "/Users/macuser/Documents/wh_visits_2022_01_to_05_cleaned.csv"
visits_school.to_csv(clean_path, index=False)

# ---- Visuals (6) ----
out_paths = []
def save_fig(name):
    p = f"/Users/macuser/Documents/{name}.png"
    plt.savefig(p, dpi=200, bbox_inches="tight")
    out_paths.append(p)
    plt.close()

# Visual 1: Line chart of weekday visit volume over time 
weekly = (
    visits_school.dropna(subset=["visit_start"])
    .groupby(pd.Grouper(key="visit_start", freq="W-MON"))
    .size()
    .rename("visits")
    .reset_index()
    .sort_values("visit_start")
)

plt.figure()
plt.plot(
    weekly["visit_start"],
    weekly["visits"],
    marker="o",
    markersize=3
)

plt.title("How busy White House visit traffic is, week by week (Jan–May 2022)")
plt.xlabel("Month")
plt.ylabel("Number of recorded visits")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

plt.tight_layout()
save_fig("vis1_line_weekly_volume")

# Visual 2: Bar chart of busyness by arrival hour (TOA-based when available)
hour_counts = (
    visits_school.dropna(subset=["visit_hour"])
    .groupby("visit_hour")
    .size()
    .reindex(range(8, 18), fill_value=0)
)
plt.figure()
plt.bar(hour_counts.index.astype(str), hour_counts.values)
plt.title("Which arrival hours are busiest? (Mon–Fri)")
plt.xlabel("Arrival hour")
plt.ylabel("Number of recorded visits")
save_fig("vis2_bar_busyness_by_hour")

# Visual 3: Stacked bar (weekday x time blocks)
weekday_order = wk
block_order = ["Early (before 10am)", "Late morning (10–11:59)", "Midday (12–1:59)", "Afternoon (2–3:59)", "Late day (4pm+)"]
wd_tb = (
    visits_school.dropna(subset=["visit_dow","time_block"])
    .groupby(["visit_dow","time_block"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=weekday_order, columns=block_order, fill_value=0)
)
bottom = np.zeros(len(wd_tb))
plt.figure()
for col in wd_tb.columns:
    plt.bar(wd_tb.index, wd_tb[col].values, bottom=bottom, label=col)
    bottom += wd_tb[col].values
plt.title("Best 'quiet windows' by weekday (stacked by time block)")
plt.xlabel("Weekday")
plt.ylabel("Number of recorded visits")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Time block", bbox_to_anchor=(1.02, 1), loc="upper left")
save_fig("vis3_stacked_weekday_timeblocks")

# Visual 4: Bar chart cancellation rate by weekday (risk management for admins)
cancel_by_wd = (
    visits_school.dropna(subset=["visit_dow"])
    .groupby("visit_dow")["canceled"]
    .mean()
    .reindex(weekday_order)
)
plt.figure()
plt.bar(cancel_by_wd.index, cancel_by_wd.values * 100)
plt.title("Which weekdays have more cancellations? (lower is safer)")
plt.xlabel("Weekday")
plt.ylabel("Canceled appointments (%)")
plt.xticks(rotation=25, ha="right")
save_fig("vis4_bar_cancellation_by_weekday")

# Visual 5: Scatterplot (time): lead days vs arrival hour, sized by group size (no custom colors)
sc = visits_school.dropna(subset=["lead_days","visit_hour"]).copy()
sc = sc[sc["lead_days"].between(0, 365)]
sizes = None
if "TOTAL_PEOPLE" in sc.columns:
    # size scaling for readability (cap extremes)
    gp = sc["TOTAL_PEOPLE"].fillna(1).clip(1, 200)
    sizes = (gp / gp.max()) * 80 + 10
plt.figure()
plt.scatter(sc["lead_days"], sc["visit_hour"], alpha=0.25, s=sizes)
plt.title("Planning tip: how far ahead visits are booked (by arrival hour)")
plt.xlabel("Days between booking and visit")
plt.ylabel("Arrival hour")
plt.yticks(range(8, 18))
save_fig("vis5_scatter_leadtime_vs_hour")

# Visual 6: Step chart cumulative share of arrivals by hour (easy 'pick earlier/later' story)
cdf_counts = (
    visits_school.dropna(subset=["visit_hour"])
    .groupby("visit_hour")
    .size()
    .reindex(range(8, 18), fill_value=0)
)
cdf = cdf_counts.cumsum() / cdf_counts.sum() if cdf_counts.sum() else cdf_counts
plt.figure()
plt.step(cdf.index, cdf.values, where="mid")
plt.title("Cumulative share of arrivals by hour (find the least crowded slice)")
plt.xlabel("Arrival hour")
plt.ylabel("Cumulative share of recorded visits")
plt.ylim(0, 1.05)
save_fig("vis6_step_cdf_by_hour")

(clean_path, visits_school.shape, out_paths)

('/Users/macuser/Documents/wh_visits_2022_01_to_05_cleaned.csv',
 (19551, 37),
 ['/Users/macuser/Documents/vis1_line_weekly_volume.png',
  '/Users/macuser/Documents/vis2_bar_busyness_by_hour.png',
  '/Users/macuser/Documents/vis3_stacked_weekday_timeblocks.png',
  '/Users/macuser/Documents/vis4_bar_cancellation_by_weekday.png',
  '/Users/macuser/Documents/vis5_scatter_leadtime_vs_hour.png',
  '/Users/macuser/Documents/vis6_step_cdf_by_hour.png'])

### Audience, Purpose, Medium, and Design Choices
The primary audience for this analysis is school administrators and trip planners responsible for organizing student visits to the White House. This audience is generally time-constrained, operationally focused, and may not have deep technical or statistical training. As a result, the purpose of the project is decision support, not exploration: to clearly identify when school visits are least likely to be disrupted by cancellations, crowding, or last-minute schedule changes. The central message is actionable—visit midweek (Tuesday or Wednesday), arrive earlier in the day, and plan several weeks in advance—allowing administrators to make practical scheduling decisions with greater confidence.

The chosen medium is a single-page infographic, which is appropriate for this audience and purpose. Rather than presenting raw tables or dense analysis, the infographic combines multiple simple charts that together tell a cohesive story. Line charts show weekly visit volume trends, bar charts compare cancellations by weekday and crowding by arrival hour, and cumulative plots illustrate how congestion increases later in the day. This combination allows readers to scan the page quickly while still understanding the evidence behind each recommendation.

Design choices prioritize clarity, hierarchy, and accessibility. Neutral color palettes and consistent axes reduce cognitive load, while annotations translate statistical patterns into plain-language insights. The final recommendation box at the bottom reinforces the call to action, ensuring that readers who skim still retain the key takeaways.

### Ethical Considerations

Several transformations were applied to the data, including aggregation by week, weekday, and arrival hour, as well as filtering to a specific time window (January–May 2022). These changes were necessary to reduce noise and focus the analysis on school-year travel patterns, but they also introduce assumptions. For example, it is assumed that historical crowding and cancellation patterns are reasonably stable over time and that the selected months are representative of typical school trips.

The data was sourced from publicly available White House visitor records, which minimizes legal and regulatory concerns and ensures ethical acquisition. No personally identifiable information was used; analysis was conducted only at an aggregate level. However, there is a risk that simplified visualizations could imply causation rather than correlation—for example, suggesting that Tuesdays “cause” fewer cancellations rather than being historically associated with them.

To mitigate these risks, the infographic explicitly references the data source and time period and frames conclusions as risk-reducing recommendations rather than guarantees. Clear labeling, transparent filtering, and conservative language help ensure that the visualization informs decision-making without overstating certainty or obscuring limitations.